In [ ]:
import os.path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', "no-latex"])

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

In [ ]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

In [ ]:
interested_locations = {
    "London": {},
    # "TLI3": {},
    # "TLI4": {},
    # "TLI5": {},
    # "TLI6": {},
    # "TLI7": {},
    "TLH1": {},
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD3":{},
    "TLD4":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE3":{},
    "TLE4":{},
}

In [ ]:
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region.reset_index(),
        "substation": interested_substations.reset_index()
    }

# Grid Points

In [ ]:
from SpatialAllocation.utils.GenerateGrid import generate_grid
from shapely.ops import unary_union
import pickle

In [ ]:
def get_color(category):
    if category == 'industrial':
        return 'red'
    elif category == 'residential':
        return kitblue
    elif category == 'commercial':
        return 'yellow'
    elif category == 'agricultural':
        return kitgreen
    else:
        return grey30

In [ ]:
def calculate_step_size(target_points, regions):
    """
    Calculate an appropriate grid step size
    :param target_points: Target number of points
    :param regions: GeoDataFrame of regions
    :return: Appropriate grid step size
    """
    regions = regions.to_crs("EPSG:3857")
    boundary_polygon_m = unary_union(regions["geometry"])
    minx, miny, maxx, maxy = boundary_polygon_m.bounds
    box_area = (maxx - minx) * (maxy - miny)
    polygon_area = boundary_polygon_m.area
    area_ratio = polygon_area / box_area
    adjusted_target = target_points / area_ratio
    ideal_step_size = np.sqrt(box_area / adjusted_target)
    step_size_m = np.floor(ideal_step_size / 10) * 10
    return max(10, step_size_m)

In [ ]:
for key in interested_locations.keys():
    grid_gdf_path = f"./results/intermediate/GridPoint/{key}_grid_points.pickle"
    print("Processing interested location:", key)
    if os.path.exists(grid_gdf_path):
        with open(grid_gdf_path, "rb") as f:
            grid_gdf, step_size_m = pickle.load(f)
            print(f"step_size_m for {key} is {step_size_m} m")
    else:
        step_size_m = calculate_step_size(target_points=200000, regions=interested_locations[key]["region"].copy())
        # step_size_m = 100
        print(f"Calculated step_size_m for {key} is {step_size_m} m")
        grid_gdf, numerical_col_names, categorical_col_members = generate_grid(interested_locations[key]["region"].copy(), step_size_m=step_size_m, with_tags={"landuse":True})
        print(grid_gdf.shape)

        grid_gdf['color'] = grid_gdf['landuse'].apply(get_color)
        with open(grid_gdf_path, "wb") as f:
            pickle.dump([grid_gdf, step_size_m], f)
    interested_locations[key]["grid"] = grid_gdf
    print(f"grid_gdf:{grid_gdf.shape}")
    interested_locations[key]["step_size_m"] = step_size_m

In [ ]:
grid_gdf

In [ ]:
# for i, key in enumerate(interested_locations.keys()):
#     fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.5), dpi=300)  # Double-column width
#     interested_locations[key]["grid"].plot(ax=ax, color=interested_locations[key]["grid"]["color"], markersize=0.2, aspect='equal', rasterized=True)
#     ax.tick_params(axis='x', labelsize=6)  # Adjust x-axis tick label size
#     ax.tick_params(axis='y', labelsize=6)
#     ax.set_xlabel(r'Longitude [$^\circ$E]', fontsize=8)
#     ax.set_ylabel(r'Latitude [$^\circ$N]', fontsize=8)
#     # ax.set_title('Original Surfaces and Points', fontsize=8)
#     # Add legend
#     legend_elements = [
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=6, label='Industrial'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=kitblue, markersize=6, label='Residential'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', markersize=6, label='Commercial'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=kitgreen, markersize=6, label='Agricultural'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=6, label='Transportation'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=grey30, markersize=6, label='Other')
#     ]
#
#     # Place the legend below the x-axis
#     ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.2),
#               ncol=3, fontsize=6)
#
#     plt.tight_layout()
#     # plt.savefig(f'./results/pictures/Figure_interested_region_{key}_with_landuse.pdf', dpi=300, bbox_inches='tight')
#     plt.show()

# Voronoi Diagramm

In [ ]:

import os
from SpatialAllocation.voronoi.clustering import do_clustering
from SpatialAllocation.utils import assign_color

In [ ]:
from SpatialAllocation.voronoi.core.SimpleVoronoi import simple_voronoi

In [ ]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/voronoi_{interested_location}_with_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    substations_key["cluster_label"] = range(len(substations_key))

    if os.path.exists(path):
        print(f"Voronoi grid for {interested_location} with subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} voronoi with subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns={"ITL3": "ITL3"})
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["voronoi_with_ITL3"] = [voronoi_gdf, substations_key]
    print("#" * 50)

In [ ]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/clustered_voronoi_{interested_location}_with_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]

    if os.path.exists(path):
        print(f"Clustered voronoi grid for {interested_location} with subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} clustered voronoi with subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns={"ITL3": "ITL3"})
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["clustered_voronoi_with_ITL3"] = [voronoi_gdf, substations_key]
    print("#" * 50)

In [ ]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/voronoi_{interested_location}_without_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    substations_key["cluster_label"] = range(len(substations_key))

    if os.path.exists(path):
        print(f"Voronoi grid for {interested_location} without subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} voronoi without subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns=None)
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["voronoi"] = [voronoi_gdf, substations_key]
    print("#" * 50)

In [ ]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/clustered_voronoi_{interested_location}_without_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]

    if os.path.exists(path):
        print(f"Clustered voronoi grid for {interested_location} without subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} clustered voronoi without subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns=None)
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["clustered_voronoi"] = [voronoi_gdf, substations_key]
    print("#" * 50)

In [ ]:
import pickle
import os
from ClusterBasedVoronoi.voronoi import build_model
from ClusterBasedVoronoi.clustering import do_clustering
from ClusterBasedVoronoi.utils import assign_color

for key in interested_locations.keys():
    path = f"./results/intermediate/Pyomo/hdbscan_voronoi_result_british_{key}.pkl"
    substations_key = interested_locations[key]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]
    if os.path.exists(path):
        with open(path, "rb") as file:
            voronoi_gdf = pickle.load(file)
    else:
        model, voronoi_gdf, grid_points = build_model(interested_locations[key]["region"], substations_key, interested_locations[key]["step_size_m"], weight="equal", method="civd")
        with open(path, "wb") as file:
            pickle.dump(voronoi_gdf, file)

    interested_locations[key]["cluster_voronoi_hdbscan"] = [voronoi_gdf, substations_key]

In [ ]:
# First, the data needs to be adapted because the original code structure is different
for key in interested_locations.keys():
    regions = interested_locations[key]["region"].copy()

    # Calculate residential demand (based on population)
    # total_england_population = ITL3_region.population.sum()
    # regions["residential_percent"] = regions["population"] / total_england_population  # 28.79% is the residential energy consumption share

    # Calculate demand for each industry (based on GVA data, which are already proportions in ITL3_region)
    # regions["industrial_percent"] = regions["industrial_gva"]
    # regions["commercial_percent"] = regions["commercial_gva"]
    # regions["agricultural_percent"] = regions["agricultural_gva"]
    # regions["others_percent"] = regions["others_gva"]

    # Normalize
    total_percent = (regions["residential_percent"] + regions["industrial_percent"] +
                    regions["commercial_percent"] + regions["agricultural_percent"] +
                    regions["others_percent"])


    regions["residential_percent"] = regions["residential_percent"] / total_percent
    regions["industrial_percent"] = regions["industrial_percent"] / total_percent
    regions["commercial_percent"] = regions["commercial_percent"] / total_percent
    regions["agricultural_percent"] = regions["agricultural_percent"] / total_percent
    regions["others_percent"] = regions["others_percent"] / total_percent

    # Calculate demand for each type
    regions["residential_demand"] = regions["Demand (MVA)"] * regions["residential_percent"]
    regions["industrial_demand"] = regions["Demand (MVA)"] * regions["industrial_percent"]
    regions["commercial_demand"] = regions["Demand (MVA)"] * regions["commercial_percent"]
    regions["agricultural_demand"] = regions["Demand (MVA)"] * regions["agricultural_percent"]
    regions["others_demand"] = regions["Demand (MVA)"] * regions["others_percent"]

    interested_locations[key]["regions_all"] = regions

In [ ]:
regions

In [ ]:
# Assign demand to each grid point
for key in interested_locations.keys():
    grid_gdf_path = f"./results/intermediate/GridPoint/{key}_grid_points.pickle"
    with open(grid_gdf_path, "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    regions = interested_locations[key]["regions_all"].copy()

    # --- Preparation: Spatial join ---
    # Need to assign demand based on the ITL3 region each grid point belongs to
    # First perform a spatial join to find which ITL3 region each grid point falls in
    # Note: Select the ITL3 column from regions to ensure it can be joined to grid_gdf
    grid_with_regions = grid_gdf.copy()

    # =================================================================================
    # Option 1: Evenly distribute demand by [landuse type] within each region (original logic)
    # =================================================================================

    # 1. Count the number of grid points for each landuse type within each ITL3 region
    landuse_counts = grid_with_regions.groupby(['ITL3', 'landuse']).size()

    # 2. Create demand mapping (ITL3, landuse) -> demand
    demand_map = {}
    landuse_types = ['residential', 'industrial','commercial', 'agricultural', 'others']
    for _, row in regions.iterrows():
        itl3 = row["ITL3"]
        for landuse_type in landuse_types:
            demand_map[(itl3, landuse_type)] = row[f"{landuse_type}_demand"]

    # 3. Assign demand to each grid point based on its landuse type
    def assign_landuse_demand(row):
        # Use .get() to handle cases where a region has no grid points of a certain landuse type
        count = landuse_counts.get((row['ITL3'], row['landuse']), 1)
        total_demand = demand_map.get((row['ITL3'], row['landuse']), 0)
        return total_demand / count if count > 0 else 0

    grid_with_regions['landuse_demand'] = grid_with_regions.apply(assign_landuse_demand, axis=1)

    # =================================================================================
    # Option 2: Evenly distribute total demand directly across the entire ITL3 region (new requirement)
    # =================================================================================

    # 1. Calculate the total demand for each ITL3 region
    demand_cols = [f"{landuse_type}_demand" for landuse_type in landuse_types]
    regions['total_ITL3_demand'] = regions[demand_cols].sum(axis=1)
    # Create mapping: ITL3 -> total_demand
    total_demand_map = regions.set_index('ITL3')['total_ITL3_demand'].to_dict()

    # 2. Count the number of grid points in each ITL3 region
    # Create mapping: ITL3 -> total_grid_points
    total_grid_count_map = grid_with_regions.groupby('ITL3').size().to_dict()

    # 3. Distribute total demand evenly to all points in the region (using .map() for better efficiency)
    # Use Series.map() to map values from a dictionary
    total_demand_per_grid = grid_with_regions['ITL3'].map(total_demand_map)
    total_count_per_grid = grid_with_regions['ITL3'].map(total_grid_count_map)

    # Calculate average and handle potential division-by-zero or NaN issues
    grid_with_regions['average_demand'] = (total_demand_per_grid / total_count_per_grid).fillna(0)
    # print(key, 1/total_count_per_grid.unique())

    # --- Data update ---
    # Keep both demand allocation methods and original demand columns in the final result
    final_columns = [
        'geometry', 'landuse', 'color',
        'landuse_demand', 'average_demand', "Demand (MVA)"
    ]
    # Filter out columns that don't exist in the original data to prevent errors
    final_columns = [col for col in final_columns if col in grid_with_regions.columns]

    interested_locations[key]["grid"] = grid_with_regions[final_columns]

In [ ]:
for key in interested_locations.keys():
    for voronoi in ["voronoi", "voronoi_with_ITL3", "clustered_voronoi", "clustered_voronoi_with_ITL3"]:
        voronoi_gdf = interested_locations[key][voronoi][0].copy()
        substations_key = interested_locations[key][voronoi][1].copy()
        grid_gdf = interested_locations[key]["grid"].copy()

        # --- FIX START ---
        # The previous .apply method was inefficient and could cause data loss due to boundary issues, leading to total demand mismatch.
        # Using spatial join is more efficient and accurate for this computation.

        # 1. Use sjoin to join grid points to their containing Voronoi polygons.
        #    'inner' ensures only points within polygons are kept, predicate='within' defines the spatial relationship.
        grid_gdf = grid_gdf.to_crs("EPSG:3857")
        voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
        points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
        points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]

        # 2. Group by polygon index (i.e., cluster_label) and sum the 'landuse_demand' of all points within each polygon.
        #    'index_right' is the column representing the right-side (polygon) DataFrame index after the sjoin operation.
        demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['landuse_demand'].sum()

        # 3. Map the computed total demand back to voronoi_gdf.
        #    Use .reindex to ensure all polygons are included; polygons containing no points will have demand set to 0.
        voronoi_gdf['landuse_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
        # --- FIX END ---

        # Allocation logic remains unchanged: evenly distribute each polygon's total demand among all substations within it.
        counts = substations_key["cluster_label"].value_counts()
        substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "landuse_demand"]/counts[x])
        interested_locations[key][voronoi][1] = substations_key
        interested_locations[key][voronoi][0]= voronoi_gdf

In [ ]:
grid_gdf

In [ ]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["cluster_voronoi_hdbscan"][0].copy()
    substations_key = interested_locations[key]["cluster_voronoi_hdbscan"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # Apply the same efficient and accurate spatial join aggregation method for the civd_gpm model.
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['landuse_demand'].sum()
    voronoi_gdf['landuse_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "landuse_demand"]/counts[x])
    interested_locations[key]["cluster_voronoi_hdbscan"][1] = substations_key
    interested_locations[key]["cluster_voronoi_hdbscan"][0]= voronoi_gdf

In [ ]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["cluster_voronoi_hdbscan"][0].copy()
    substations_key = interested_locations[key]["cluster_voronoi_hdbscan"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # Apply the same efficient and accurate spatial join aggregation method for the civd_gpm model.
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['average_demand'].sum()
    voronoi_gdf['average_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "average_demand"]/counts[x])
    interested_locations[key]["cluster_voronoi_hdbscan_avg"] = [voronoi_gdf, substations_key]

In [ ]:
grid_gdf

In [ ]:
voronoi_gdf

In [ ]:
points_in_voronoi

In [ ]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["voronoi"][0].copy()
    substations_key = interested_locations[key]["voronoi"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # Apply the same spatial join fix logic to 'average_demand' to ensure total demand conservation for the simple_voronoi method.
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['average_demand'].sum()
    voronoi_gdf['average_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "average_demand"]/counts[x])
    interested_locations[key]["simple_voronoi"] = [voronoi_gdf, substations_key]

In [ ]:
for key in interested_locations.keys():
    # --- Logic for 'average' (ITL3-level) ---
    # Logic: Within each ITL3 region, evenly distribute the total demand among all substations in that region.
    region_avg_demand = interested_locations[key]["region"].copy()
    substations_key = interested_locations[key]["voronoi"][1].copy()

    # Count the number of substations in each region by ITL3
    substation_counts = substations_key.groupby('ITL3').size().reset_index(name='substation_count')

    # Merge the substation counts back into the region demand data
    region_avg_demand = region_avg_demand.merge(substation_counts, on='ITL3', how='left')
    region_avg_demand['substation_count'] = region_avg_demand['substation_count'].fillna(1) # Fill with 1 for regions without substations to avoid division by zero

    # Calculate the average demand for each ITL3 region
    region_avg_demand['avg_demand'] = region_avg_demand['Demand (MVA)'] / region_avg_demand['substation_count']

    # Merge the computed average demand values back into the substation data
    substations_key = substations_key.merge(region_avg_demand[['ITL3', 'avg_demand']], on='ITL3', how='left')
    substations_key.rename(columns={'avg_demand': 'ITL3_average'}, inplace=True)

    # --- Start of new 'Global Average' logic ---
    # Logic: Treat the entire study area as a whole and evenly distribute total demand among all substations.

    # 1. Calculate the total demand in the area (Total Demand). We sum the actual demand directly from the data for the most accurate total.
    total_global_demand = substations_key['Demand (MVA)'].sum()
    # 2. Count the total number of substations in the area
    total_substation_count = len(substations_key)
    # 3. Calculate the global average and assign it to each substation
    substations_key['ITL2_average'] = total_global_demand / total_substation_count
    # --- End of 'Global Average' logic ---

    # Store the DataFrame with all newly computed columns back
    interested_locations[key]["voronoi"][1] = substations_key

In [ ]:
# rows = ["simple_voronoi", "voronoi", "voronoi_with_ITL3", "clustered_voronoi", "clustered_voronoi_with_ITL3", "ITL3_average", "ITL2_average", "civd_gpm"]
rows = ["simple_voronoi", "voronoi", "civd", "civd_gpm", "ITL3_average", "ITL2_average"]
cols = interested_locations.keys()
corr_matrix = pd.DataFrame(index=rows, columns=cols)
corr_max_matrix = pd.DataFrame(index=rows, columns=cols)
rmse_matrix = pd.DataFrame(index=rows, columns=cols)
over_matrix = pd.DataFrame(index=rows, columns=cols)
mae_matrix = pd.DataFrame(index=rows, columns=cols)

In [ ]:
# for key in ["TLI3", "TLH1"]:
for key in interested_locations.keys():
    substations_key = interested_locations[key]["voronoi"][1].copy()
    substations_key["simple_voronoi"] = interested_locations[key][f"simple_voronoi"][1]["voronoi"].copy()
    substations_key["voronoi_with_ITL3"] = interested_locations[key]["voronoi_with_ITL3"][1]["voronoi"].copy()
    substations_key["clustered_voronoi"] = interested_locations[key]["clustered_voronoi"][1]["voronoi"].copy()
    substations_key["clustered_voronoi_with_ITL3"] = interested_locations[key]["clustered_voronoi_with_ITL3"][1]["voronoi"].copy()
    substations_key[f"civd_gpm"] = interested_locations[key]["cluster_voronoi_hdbscan"][1]["voronoi"].copy()
    substations_key[f"civd"] = interested_locations[key]["cluster_voronoi_hdbscan_avg"][1]["voronoi"].copy()

    for row in rows:
        corr_matrix.loc[row, key] = substations_key["Demand (MVA)"].corr(substations_key[row])
        corr_max_matrix.loc[row, key] = substations_key["Firm Capacity (MVA)"].corr(substations_key[row])
        rmse_matrix.loc[row, key] = np.sqrt(((substations_key["Demand (MVA)"] - substations_key[row]) ** 2).mean())
        over_matrix.loc[row, key] = len(substations_key[substations_key[row] > substations_key["Firm Capacity (MVA)"]])
        mae_matrix.loc[row, key] = np.mean(np.abs(substations_key["Demand (MVA)"] - substations_key[row]))

In [ ]:
substations_key[["Demand (MVA)", "simple_voronoi", "voronoi", "voronoi_with_ITL3", "civd", "civd_gpm", "ITL3_average", "ITL2_average"]].sum()

In [ ]:
corr_matrix

In [ ]:
rmse_matrix

In [ ]:
mae_matrix

In [ ]:
with open("./results/intermediate/static_matrix.pickle", "wb") as f:
    pickle.dump({
        "corr_matrix": corr_matrix,
        "rmse_matrix": rmse_matrix,
        "over_matrix": over_matrix,
        "mae_matrix": mae_matrix
    }, f)